In [1]:
import json
from pathlib import Path
import pickle

from rich.progress import track

import pandas as pd

In [2]:
EXPERIMENT_NAME = "base-experiment"
ANALYSIS_BASE = "./analysis/paper-data/"
INCLUDE_EWMA = False
INCLUDE_GRADIENTS = False

CROSS_VALIDATIONS_PATH = f"./cross-validations-{EXPERIMENT_NAME}"


In [3]:
def filter_dataframe_for_current_experiment(data_flattened):
    df = data_flattened.loc[(slice(None), slice(None), INCLUDE_GRADIENTS, INCLUDE_EWMA), :]
    df.index = df.index.droplevel([2, 3])
    df = df.sort_values(by=['percentage', 'model'])

    def float_or_list_to_tuple(x):
        if isinstance(x, list):
            return tuple(x)
        return x

    df["hidden_layer_sizes"] = df['hidden_layer_sizes'].apply(float_or_list_to_tuple)
    return df

In [4]:

def parse_files(cross_validations_path, seed):
    """
    Parse files and use pd.json_normalize to flatten the json.
    """
    files = Path(ANALYSIS_BASE).joinpath(f"{cross_validations_path}-{seed}").glob("**/*.json")
    results = pd.DataFrame()
    files = list(files)

    for file in track(
        files,
        description="Loading hyperparameters and "
        "performance data from file to DataFrame",
    ):
        with open(file) as f:
            data = json.load(f)
            original_target = data["original_target"]
            original_dict = json.loads(Path(original_target).read_text())

            f1_scores = pd.json_normalize(data, "f1_scores")
            normalized_data = pd.json_normalize(original_dict)
            normalized_data = pd.concat(
                [normalized_data] * len(f1_scores), ignore_index=True
            )
            normalized_data["metrics.f1_score"] = f1_scores
            normalized_data["paths.model_path"] = data["model_path"]
            results = pd.concat([results, normalized_data])

    results = results.set_index(
        ["percentage", "model", "use_gradient", "use_ewma"]
    )
    results.columns = pd.MultiIndex.from_arrays(
        zip(*results.columns.str.split(".", expand=True))
    )
    results = results.sort_index(axis=1)
    results = results.drop(columns=['k_fold', 'preprocessing'])
    results = results.droplevel(0, axis=1)

    results = filter_dataframe_for_current_experiment(results)

    return results

In [5]:
def pick_best_hyperparameters_from_k_folds(k_folds_data):

    PERCENTAGES = [1, 5, 10, 15, 20, 25, 30, 35, 40 ]
    MODELS = ['RF', 'DUM', 'ET', 'AdaBoost']
    FILTER_MODELS = True

    # This is a bit hacky, but I add model_path here because its only always the same
    # so it doesn't have effect on the groupby.
    # Would be better to `df.merge` though.
    group = k_folds_data.groupby(['percentage', 'model', 'C', 'hidden_layer_sizes', 'kernel', 'learning_rate', 'max_depth', 'n_estimators', 'alpha', 'model_path'], dropna=False)
    summary = group['f1_score'].agg(['mean', 'std'])

    idx = summary.groupby(['percentage', 'model']).idxmax()

    best_hyperparams = summary.loc[idx['mean'], :]

    best_models_per_percentage_and_type = best_hyperparams[best_hyperparams.index.get_level_values('percentage').isin(PERCENTAGES)]
    if FILTER_MODELS:
        best_models_per_percentage_and_type =  best_models_per_percentage_and_type[best_models_per_percentage_and_type.index.get_level_values('model').isin(MODELS)]
    return best_models_per_percentage_and_type



In [6]:
def load_classifiers_from_best_models(best_models_per_percentage_and_type):
    classifiers = {}
    for _, row in best_models_per_percentage_and_type.reset_index().iterrows():
        
        with open(row['model_path'], 'rb') as f:
            instantiated_model = pickle.load(f)
        classifiers[(row['percentage'], row['model'])] = instantiated_model
    return classifiers

In [39]:
data_for_all_seeds = []
for i in range(1,11):

    current_seed = {
        'seed': i
    }

    data = parse_files(Path(CROSS_VALIDATIONS_PATH), seed=i)
    current_seed['k_fold'] = data
    
    current_seed['best_models'] = pick_best_hyperparameters_from_k_folds(data)
    current_seed['classifiers'] = load_classifiers_from_best_models(current_seed['best_models'])

    TEST_DATASET_PATH = Path(f'./analysis/paper-data/{EXPERIMENT_NAME}-{i}.pkl_test.pkl')
    with open(TEST_DATASET_PATH, 'rb') as f:
        TEST_DATASET = pickle.load(f)
        TEST_DATASET = TEST_DATASET
        
    TRAIN_DATASET_PATH = Path(f'./analysis/paper-data/{EXPERIMENT_NAME}-{i}.pkl_train.pkl')
    with open(TRAIN_DATASET_PATH, 'rb') as f:
        TRAIN_DATASET = pickle.load(f)
        TRAIN_DATASET = TRAIN_DATASET

    current_seed['train_data'] = TRAIN_DATASET
    current_seed['test_data'] = TEST_DATASET
    
    data_for_all_seeds += [current_seed]

Output()

Output()

Output()

Output()

Output()

Output()

Output()

Output()

Output()

Output()

# Utility

In [40]:
from sklearn.preprocessing import MaxAbsScaler
def preprocess(dataframe, scaler=None):
    result, _ = dataframe.drop(columns=['mzn', 'dzn'], axis=1), dataframe['mzn']

    # Drop any columns that contain the text 'ewma' or 'gradient'
    result = result.drop(columns=result.columns[result.columns.str.contains('ewma|gradient')], axis=1)
    result = result.drop(result.columns[result.nunique() == 1], axis=1)

    
    scaler = MaxAbsScaler().fit(result)    
    # result = pd.DataFrame(scaler.transform(result), columns=result.columns, index=result.index)

    return result




In [41]:
def combine_seed_data_into_single_dataframe_opinionated(total_data):
    dfs = []
    for seed in total_data:
        best_models_copy = seed['best_models'].copy()
        best_models_copy['seed'] = seed['seed']
        dfs.append(best_models_copy)
    return pd.concat(dfs)

In [42]:
data_for_all_seeds

[{'seed': 1,
  'k_fold':                        C  alpha hidden_layer_sizes kernel  learning_rate  \
  percentage model                                                           
  1          AdaBoost  NaN    NaN                NaN    NaN            0.1   
             AdaBoost  NaN    NaN                NaN    NaN            0.1   
             AdaBoost  NaN    NaN                NaN    NaN            0.1   
             AdaBoost  NaN    NaN                NaN    NaN            0.1   
             AdaBoost  NaN    NaN                NaN    NaN            0.1   
  ...                  ...    ...                ...    ...            ...   
  40         SVM       0.1    NaN                NaN    rbf            NaN   
             SVM       0.1    NaN                NaN    rbf            NaN   
             SVM       0.1    NaN                NaN    rbf            NaN   
             SVM       0.1    NaN                NaN    rbf            NaN   
             SVM       0.1    NaN        

# Now here we can split off: our base experiment.

In [52]:
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, balanced_accuracy_score, precision_score, recall_score
import numpy as np
import matplotlib
matplotlib.use("pgf")
matplotlib.rcParams.update({
    # "pgf.texsystem": "lualatex",
    'font.family': 'serif',
    'text.usetex': False,
    'pgf.rcfonts': False,
})

In [44]:
# temp = data_for_all_seeds[1]['k_fold']
# group = temp.groupby(['percentage', 'model', 'C', 'hidden_layer_sizes', 'kernel', 'learning_rate', 'max_depth', 'n_estimators', 'alpha', 'model_path'], dropna=False)
# summary = group['f1_score'].agg(['mean', 'std'])

In [76]:
from collections import defaultdict
def fit_and_test_model(test_data, train_data, instantiated_model):
    test_x, test_y = test_data.drop(['solved_within_time_limit'], axis=1), test_data['solved_within_time_limit']
    train_x, train_y = train_data.drop(['solved_within_time_limit'], axis=1), train_data['solved_within_time_limit']

    test_x = preprocess(test_x)
    train_x = preprocess(train_x)
    
    instantiated_model.fit(train_x, train_y)
    
      # Predict on the test set using all features
    y_pred = instantiated_model.predict(test_x)
    
    # Compute the F1 score
    score = f1_score(test_y, y_pred, average='binary')
    balanced = balanced_accuracy_score(test_y, y_pred)
    precision = precision_score(test_y, y_pred)
    recall = recall_score(test_y, y_pred)
    return (balanced, precision, recall, score)



def run_best_models_on_test_set(all_data):
    results = defaultdict(list)
    for seed, i in enumerate(all_data):
        test_data = seed['test_data']
        train_data = seed['train_data']
        for k, instantiated_model in seed['classifiers'].items():
            percentage, model = k
            scores = fit_and_test_model(test_data=test_data[percentage], train_data=train_data[percentage], instantiated_model=instantiated_model)
            results[(k, i)]= scores
    # return pd.DataFrame.from_records([{'percentage': k[0], 'model': k[1], 'mean': np.mean(v), 'std': np.std(v)} for k, v in dict(results).items()])
    return pd.DataFrame.from_records([{'percentage': k[0][0] / 2, 'model': k[0][1], 'seed': k[1], 'balanced': v[0], 'precision': v[1], 'recall': v[2], 'f1': v[3]} for k, v in dict(results).items()])
            

In [77]:
f1s = run_best_models_on_test_set(data_for_all_seeds)

TypeError: 'int' object is not subscriptable

In [73]:
f1s.balanced.std()

TypeError: setting an array element with a sequence.

In [62]:
print(f1s.to_latex())

\begin{tabular}{lrlrrrr}
\toprule
 & percentage & model & balanced & precision & recall & f1 \\
\midrule
0 & 0.500000 & AdaBoost & 0.809391 & 0.839919 & 0.858103 & 0.871852 \\
1 & 0.500000 & DUM & 0.384374 & 0.422772 & 0.440269 & 0.460915 \\
2 & 0.500000 & ET & 0.821848 & 0.913376 & 0.893882 & 0.906900 \\
3 & 0.500000 & RF & 0.868040 & 0.904042 & 0.916435 & 0.900539 \\
4 & 2.500000 & AdaBoost & 0.797890 & 0.816860 & 0.860569 & 0.773151 \\
5 & 2.500000 & DUM & 0.296096 & 0.495518 & 0.443140 & 0.357759 \\
6 & 2.500000 & ET & 0.868621 & 0.865260 & 0.900628 & 0.870246 \\
7 & 2.500000 & RF & 0.879385 & 0.874691 & 0.900628 & 0.887716 \\
8 & 5.000000 & AdaBoost & 0.672412 & 0.693949 & 0.781371 & 0.735095 \\
9 & 5.000000 & DUM & 0.318061 & 0.353704 & 0.264972 & 0.241299 \\
10 & 5.000000 & ET & 0.725989 & 0.768066 & 0.820113 & 0.853747 \\
11 & 5.000000 & RF & 0.725989 & 0.739839 & 0.780498 & 0.853747 \\
12 & 7.500000 & AdaBoost & 0.765337 & 0.625839 & 0.749940 & 0.816445 \\
13 & 7.500000 & DUM 

In [64]:
f1s.f1

0     0.871852
1     0.460915
2     0.906900
3     0.900539
4     0.773151
5     0.357759
6     0.870246
7     0.887716
8     0.735095
9     0.241299
10    0.853747
11    0.853747
12    0.816445
13    0.280547
14    0.891568
15    0.873593
16    0.553517
17    0.325616
18    0.575116
19    0.609755
20    0.746606
21    0.275047
22    0.814076
23    0.723002
24    0.681667
25    0.175291
26    0.681667
27    0.669257
28    0.699254
29    0.168555
30    0.803601
31    0.773328
32    0.371622
33    0.108108
34    0.625000
35    0.626783
Name: f1, dtype: float64

In [61]:
def plot_results(results):
    plt.figure(figsize=(5.5, 3))

    
    
    for model in results['model'].unique():
        results_for_model = results[results['model'] == model]
        percentages = results_for_model['percentage']
        f1_scores_mean = results_for_model['mean']
        f1_scores_std = results_for_model['std']
        plt.errorbar(percentages, f1_scores_mean, yerr=f1_scores_std, fmt='o-', capsize=5, label=f'{model}')
        plt.fill_between(percentages, 
                        np.array(f1_scores_mean) - np.array(f1_scores_std), 
                        np.array(f1_scores_mean) + np.array(f1_scores_std), 
                        alpha=0.1)

    # plt.title('F1 Score averaged over 10 testsets of random problems over different percentages of the TL')
    plt.xlabel('Percentage of time limit')
    plt.ylabel('F1 Score')
    plt.grid(True)
    plt.ylim(0, 1)
    plt.xticks(percentages)
    plt.legend()
    plt.savefig('f1_over_time.pgf', bbox_inches='tight')
    return None
    # plt.show()
plot_results(f1s)

KeyError: 'mean'